In [1]:
import json
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# Load files
# --------------------------------------------------

with open("dataset_sample_10k.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

with open("semantic_relevance_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

meta_df = pd.DataFrame(metadata)
results_df = pd.DataFrame(results)

# --------------------------------------------------
# Merge
# --------------------------------------------------

df = results_df.merge(
    meta_df[
        [
            "dataset_id",
            "title",
            "description",
            "keywords"
        ]
    ],
    on="dataset_id",
    how="inner"
)

# --------------------------------------------------
# Output directory
# --------------------------------------------------

output_dir = Path("outlier_reports")
output_dir.mkdir(exist_ok=True)

# --------------------------------------------------
# Metrics
# --------------------------------------------------

metrics = [
    "agg_title_similarity",
    "agg_description_similarity",
    "avg_title_similarity",
    "avg_description_similarity"
]

# --------------------------------------------------
# Analyse
# --------------------------------------------------

for metric in metrics:

    print("\n" + "=" * 100)
    print(f"METRIC: {metric}")
    print("=" * 100)

    q1 = df[metric].quantile(0.25)
    q3 = df[metric].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    low_outliers = df[df[metric] < lower].copy()
    high_outliers = df[df[metric] > upper].copy()

    print(f"Q1={q1:.3f}")
    print(f"Q3={q3:.3f}")
    print(f"IQR={iqr:.3f}")
    print(f"Lower threshold={lower:.3f}")
    print(f"Upper threshold={upper:.3f}")

    print(f"\nLow outliers: {len(low_outliers)}")
    print(f"High outliers: {len(high_outliers)}")

    # --------------------------------------------------
    # Print top examples
    # --------------------------------------------------

    print("\nTOP HIGH OUTLIERS")

    for _, row in (
        high_outliers
        .sort_values(metric, ascending=False)
        .head(5)
        .iterrows()
    ):

        print("\n" + "-" * 100)
        print("TITLE:")
        print(row["title"])

        print("\nSCORE:")
        print(row[metric])

        print("\nNUM KEYWORDS:")
        print(len(row["keywords"]))

        print("\nKEYWORDS:")
        print(row["keywords"])

        print("\nDESCRIPTION:")
        print(row["description"])

    print("\nTOP LOW OUTLIERS")

    for _, row in (
        low_outliers
        .sort_values(metric)
        .head(5)
        .iterrows()
    ):

        print("\n" + "-" * 100)
        print("TITLE:")
        print(row["title"])

        print("\nSCORE:")
        print(row[metric])

        print("\nNUM KEYWORDS:")
        print(len(row["keywords"]))

        print("\nKEYWORDS:")
        print(row["keywords"])

        print("\nDESCRIPTION:")
        print(row["description"])

    # --------------------------------------------------
    # Save ALL outliers
    # --------------------------------------------------

    high_outliers["outlier_type"] = "high"
    low_outliers["outlier_type"] = "low"

    all_outliers = pd.concat(
        [high_outliers, low_outliers],
        ignore_index=True
    )

    # easier to read in csv
    all_outliers["keywords"] = all_outliers["keywords"].apply(
        lambda x: "; ".join(x)
    )

    csv_path = output_dir / f"{metric}_outliers.csv"

    all_outliers.to_csv(
        csv_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"\nSaved: {csv_path}")

C:\Users\USER\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (



METRIC: agg_title_similarity
Q1=0.308
Q3=0.558
IQR=0.250
Lower threshold=-0.067
Upper threshold=0.934

Low outliers: 2
High outliers: 7

TOP HIGH OUTLIERS

----------------------------------------------------------------------------------------------------
TITLE:
Strategic Industrial Land

SCORE:
1.0

NUM KEYWORDS:
1

KEYWORDS:
['Strategic Industrial Land']

DESCRIPTION:
Strategic Industrial Land in the London Borough of Ealing

----------------------------------------------------------------------------------------------------
TITLE:
Towns and Cities

SCORE:
1.0

NUM KEYWORDS:
1

KEYWORDS:
['Towns and Cities']

DESCRIPTION:
{{description}}

----------------------------------------------------------------------------------------------------
TITLE:
Processional Route

SCORE:
1.0

NUM KEYWORDS:
1

KEYWORDS:
['Processional Route']

DESCRIPTION:
The Processional Route in the City of London

---------------------------------------------------------------------------------------------------